# PHẦN 2: TRỰC QUAN HÓA VÀ PHÂN TÍCH DỮ LIỆU
## YÊU CẦU 1: TIỀN XỬ LÝ DỮ LIỆU (DATA PREPROCESSING)

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

# Tắt các cảnh báo không cần thiết để output sạch sẽ hơn
warnings.filterwarnings('ignore')

# 1. Cấu hình đường dẫn
DATA_DIR = 'datathon-2026-round-1/'      # Thư mục chứa file gốc
OUTPUT_DIR = 'cleaned_data/' # Thư mục lưu file sạch
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 2. Danh sách file cần xử lý
FILES = [
    'products.csv', 'customers.csv', 'promotions.csv', 'geography.csv',
    'orders.csv', 'order_items.csv', 'payments.csv', 'shipments.csv',
    'returns.csv', 'reviews.csv', 'sales.csv', 'inventory.csv', 'web_traffic.csv'
]

# 3. Định nghĩa các cột đặc thù
DATE_COLS = [
    'signup_date', 'start_date', 'end_date', 'order_date', 
    'ship_date', 'delivery_date', 'return_date', 'review_date', 
    'Date', 'snapshot_date', 'date'
]

NUMERIC_COLS = [
    'price', 'cogs', 'discount_value', 'revenue', 'payment_value', 
    'quantity', 'units_received', 'units_sold', 'shipping_fee', 'refund_amount'
]

def preprocess_data():
    for file_name in FILES:
        file_path = os.path.join(DATA_DIR, file_name)
        
        if not os.path.exists(file_path):
            print(f"⚠️ Bỏ qua: {file_name} (Không tìm thấy)")
            continue
            
        # --- BƯỚC 1: ĐỌC FILE (Xử lý lỗi Mixed Types bằng low_memory=False) ---
        df = pd.read_csv(file_path, low_memory=False)
        print(f"\n--- Đang xử lý: {file_name} ---")
        print(f"Kích thước gốc: {df.shape[0]} dòng, {df.shape[1]} cột")

        # --- BƯỚC 2: CHUẨN HÓA ĐỊNH DẠNG ---
        for col in df.columns:
            # Xử lý ngày tháng
            if col in DATE_COLS:
                df[col] = pd.to_datetime(df[col], errors='coerce')
            
            # Xử lý kiểu số
            if any(num_key in col.lower() for num_key in NUMERIC_COLS) or col in NUMERIC_COLS:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            
            # Ép kiểu String cho các cột ID để tránh lỗi mixed type khi Merge
            if 'id' in col.lower() or 'zip' in col.lower():
                df[col] = df[col].astype(str).replace(['nan', 'None', 'null'], 'Unknown')

        # --- BƯỚC 3: XỬ LÝ DÒNG TRÙNG LẶP ---
        initial_count = len(df)
        df = df.drop_duplicates()
        if len(df) < initial_count:
            print(f"✔️ Đã loại bỏ {initial_count - len(df)} dòng trùng lặp.")

        # --- BƯỚC 4: XỬ LÝ GIÁ TRỊ THIẾU (MISSING VALUES) ---
        null_info = df.isnull().sum()
        cols_with_null = null_info[null_info > 0]
        
        for col in cols_with_null.index:
            if col in DATE_COLS:
                continue # Giữ NaT cho ngày tháng để tránh sai lệch logic thời gian
            elif df[col].dtype == 'object':
                df[col] = df[col].fillna('Unknown')
            else:
                # Với các cột số lượng/tiền, fill 0 nếu null
                df[col] = df[col].fillna(0)
        
        if not cols_with_null.empty:
            print(f"✔️ Đã xử lý missing values tại: {list(cols_with_null.index)}")

        # --- BƯỚC 5: KIỂM TRA RÀNG BUỘC LOGIC (Business Rules) ---
        if 'cogs' in df.columns and 'price' in df.columns:
            invalid = df[df['cogs'] > df['price']]
            if not invalid.empty:
                print(f"❌ Cảnh báo: Có {len(invalid)} dòng vi phạm logic (cogs > price)")

        # --- BƯỚC 6: XUẤT DỮ LIỆU SẠCH ---
        output_path = os.path.join(OUTPUT_DIR, f"cleaned_{file_name}")
        df.to_csv(output_path, index=False)
        print(f"💾 File sạch lưu tại: {output_path}")

if __name__ == "__main__":
    preprocess_data()
    print("\n✅ HOÀN THÀNH TOÀN BỘ QUY TRÌNH TIỀN XỬ LÝ!")


--- Đang xử lý: products.csv ---
Kích thước gốc: 2412 dòng, 8 cột
💾 File sạch lưu tại: cleaned_data/cleaned_products.csv

--- Đang xử lý: customers.csv ---
Kích thước gốc: 121930 dòng, 7 cột
💾 File sạch lưu tại: cleaned_data/cleaned_customers.csv

--- Đang xử lý: promotions.csv ---
Kích thước gốc: 50 dòng, 10 cột
✔️ Đã xử lý missing values tại: ['applicable_category']
💾 File sạch lưu tại: cleaned_data/cleaned_promotions.csv

--- Đang xử lý: geography.csv ---
Kích thước gốc: 39948 dòng, 4 cột
💾 File sạch lưu tại: cleaned_data/cleaned_geography.csv

--- Đang xử lý: orders.csv ---
Kích thước gốc: 646945 dòng, 8 cột
💾 File sạch lưu tại: cleaned_data/cleaned_orders.csv

--- Đang xử lý: order_items.csv ---
Kích thước gốc: 714669 dòng, 7 cột
💾 File sạch lưu tại: cleaned_data/cleaned_order_items.csv

--- Đang xử lý: payments.csv ---
Kích thước gốc: 646945 dòng, 4 cột
💾 File sạch lưu tại: cleaned_data/cleaned_payments.csv

--- Đang xử lý: shipments.csv ---
Kích thước gốc: 566067 dòng, 4 cột
💾 


### 1. Kiểm tra Tổng quan (Descriptive Inspection)
Sử dụng các hàm `.shape` và `.dtypes` để xác định:
* **Quy mô dữ liệu:** Xác định tổng số dòng và số cột của từng bảng.
* **Cấu trúc:** Rà soát danh sách các thuộc tính hiện có để lập bản đồ dữ liệu (Data Mapping).

### 2. Chuẩn hóa định dạng (Type Casting)
* **Dữ liệu thời gian:** Chuyển đổi các cột ngày tháng về định dạng chuẩn bằng `pd.to_datetime`. Các giá trị lỗi sẽ được xử lý bằng `errors='coerce'` (chuyển thành `NaT`) để tránh làm gián đoạn luồng xử lý.
* **Dữ liệu số:** Ép kiểu các cột định lượng (Price, Revenue, Quantity) về dạng `numeric` để phục vụ các phép toán thống kê.

### 3. Xử lý giá trị thiếu (Missing Values)
Chiến lược xử lý được tối ưu hóa theo đặc thù kinh doanh:
* **Dữ liệu phân loại (Categorical):** Các cột như `gender` hoặc `age_group` được điền **'Unknown'** thay vì xóa bỏ, giúp bảo toàn quy mô mẫu cho các phân tích nhân khẩu học.
* **Dữ liệu số (Numerical):** Các cột chiết khấu (`discount_value`) nếu thiếu sẽ được điền **0**, tương ứng với việc đơn hàng không áp dụng khuyến mãi.

### 4. Xử lý dữ liệu trùng lặp (Duplicate Records)
Sử dụng hàm `.drop_duplicates()` để loại bỏ các bản ghi bị lặp, đảm bảo các chỉ số đo lường (KPIs) như tổng doanh thu và lượng khách hàng không bị phóng đại so với thực tế.

### 5. Kiểm tra ràng buộc Logic (Business Logic Validation)
Thực hiện kiểm tra các quy tắc cốt lõi:
* **Ràng buộc giá:** Kiểm tra điều kiện **$cogs < price$** (Giá vốn phải thấp hơn giá bán). 
* Các dòng vi phạm sẽ được hệ thống cảnh báo để hiệu chỉnh, đảm bảo tính chính xác cho các chỉ số biên lợi nhuận (Profit Margin).

### 6. Lưu trữ dữ liệu sạch (Data Export)
Tất cả kết quả được xuất ra thư mục `cleaned_data/` với tiền tố `cleaned_`. Việc này giúp tách biệt hoàn toàn dữ liệu gốc (Raw data) và dữ liệu đã qua xử lý, phục vụ tốt cho việc tái hiện kết quả (Reproducibility).